In [8]:
print("hi github")
print("hi")

hi github
hi


In [9]:
from bs4 import BeautifulSoup
soup = BeautifulSoup(page.text, "html.parser")
book_cards = soup.find_all("article", class_="product_pod")
print("the number of books found", len(book_cards))

ModuleNotFoundError: No module named 'bs4'

In [ ]:
RATING_WORDS = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}
scraped_records = []
for card in book_cards[:6]:
  title = card.h3.a["title"]
  price_gbp_str = card.find("p", class_="price_color").text.strip()
  price_gbp = float(price_gbp_str.replace('Â£', ''))

  # Correctly find the rating element using the class 'star-rating'
  rating_word = card.find("p", class_="star-rating")["class"][1]
  rating = RATING_WORDS.get(rating_word)
  scraped_records.append(
      (title, price_gbp, rating_word)
  )
  print("title\n", title, "price\n", price_gbp, "rating\n", rating_word)

title
 A Light in the Attic price
 51.77 rating
 Three
title
 Tipping the Velvet price
 53.74 rating
 One
title
 Soumission price
 50.1 rating
 One
title
 Sharp Objects price
 47.82 rating
 Four
title
 Sapiens: A Brief History of Humankind price
 54.23 rating
 Five
title
 The Requiem Red price
 22.65 rating
 One


In [ ]:
scraped_df = pd.DataFrame(scraped_records)
scraped_df.columns = ['title', 'price_gbp', 'rating_word']
scraped_df

,title,price_gbp,rating_word
0,A Light in the Attic,51.77,Three
1,Tipping the Velvet,53.74,One
2,Soumission,50.10,One
3,Sharp Objects,47.82,Four
4,Sapiens: A Brief History of Humankind,54.23,Five
5,The Requiem Red,22.65,One


In [ ]:
sample_title = scraped_df["title"].iloc[0]
response = requests.get("https://openlibrary.org/search.json", params={"title": sample_title, "limit": 1})
print("status code : ", response.status_code)
response.json()

status code :  200


{'numFound': 15,
 'start': 0,
 'numFoundExact': True,
 'num_found': 15,
 'documentation_url': 'https://openlibrary.org/dev/docs/api/search',
 'q': '',
 'offset': None,
 'docs': [{'author_key': ['OL548174A'],
   'author_name': ['Shel Silverstein'],
   'cover_edition_key': 'OL24753251M',
   'cover_i': 6806998,
   'ebook_access': 'printdisabled',
   'edition_count': 9,
   'first_publish_year': 1981,
   'has_fulltext': True,
   'ia': ['lightinatticsilv00silv', 'lightinatticsilv00silv'],
   'ia_collection': ['americana',
    'internetarchivebooks',
    'openlibrary-d-ol',
    'printdisabled',
    'stmaryscountylibrary'],
   'key': '/works/OL15843795W',
   'language': ['spa', 'eng', 'chi'],
   'public_scan_b': False,
   'title': 'A Light in the Attic'}]}

In [ ]:
import time
api_records = []
for title in scraped_df["title"]:
  response = requests.get("https://openlibrary.org/search.json", params={"title": title})
  response_json = response.json()
  print("status code : ", response.status_code)

  author_name = None
  first_publish_year = None

  if response_json and 'docs' in response_json and len(response_json['docs']) > 0:
    doc = response_json['docs'][0] # Take the first document
    if 'author_name' in doc and len(doc['author_name']) > 0:
      author_name = doc['author_name'][0] # Get the first author's name
    if 'first_publish_year' in doc:
      first_publish_year = doc['first_publish_year']

  api_records.append({
      'title': title,
      'author': author_name,
      'first_publish_year': first_publish_year
  })
  # Removed break to process all titles and added a small delay
  time.sleep(0.3)

status code :  200
status code :  200
status code :  200
status code :  200
status code :  200
status code :  200


In [ ]:
# Print details for the first record in api_records, if available
if api_records:
  first_record = api_records[0]
  print(f"Author: {first_record['author']}, Publish Year: {first_record['first_publish_year']}")
else:
  print("No API records found.")

Author: Shel Silverstein, Publish Year: 1981


In [ ]:
api_df = pd.DataFrame(api_records)
api_df

,title,author,first_publish_year
0,A Light in the Attic,Shel Silverstein,1981
1,Tipping the Velvet,Sarah Waters,1998
2,Soumission,Michel Houellebecq,2015
3,Sharp Objects,Gillian Flynn,2006
4,Sapiens: A Brief History of Humankind,BookNation,2020
5,The Requiem Red,Brynn Chapman,2016


In [ ]:
scraped_df.columns = ['title', 'price_gbp', 'rating_word']
combined_df = pd.merge(scraped_df, api_df, on="title", how="left")
combined_df

,title,price_gbp,rating_word,author,first_publish_year
0,A Light in the Attic,51.77,Three,Shel Silverstein,1981
1,Tipping the Velvet,53.74,One,Sarah Waters,1998
2,Soumission,50.10,One,Michel Houellebecq,2015
3,Sharp Objects,47.82,Four,Gillian Flynn,2006
4,Sapiens: A Brief History of Humankind,54.23,Five,BookNation,2020
5,The Requiem Red,22.65,One,Brynn Chapman,2016


In [ ]:
combined_df.to_csv("book_fair_collected.csv", index=False)

In [ ]:
reloaded_df = pd.read_csv("book_fair_collected.csv")
reloaded_df.head()

,title,price_gbp,rating_word,author,first_publish_year
0,A Light in the Attic,51.77,Three,Shel Silverstein,1981
1,Tipping the Velvet,53.74,One,Sarah Waters,1998
2,Soumission,50.10,One,Michel Houellebecq,2015
3,Sharp Objects,47.82,Four,Gillian Flynn,2006
4,Sapiens: A Brief History of Humankind,54.23,Five,BookNation,2020


In [ ]:
reloaded_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 5 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   title               6 non-null      object 
 1   price_gbp           6 non-null      float64
 2   rating_word         6 non-null      object 
 3   author              6 non-null      object 
 4   first_publish_year  6 non-null      int64  
dtypes: float64(1), int64(1), object(3)
memory usage: 372.0+ bytes


In [ ]:
reloaded_df.describe()

,price_gbp,first_publish_year
count,6.000000,6.00000
mean,46.718333,2006.00000
std,12.026490,14.60137
min,22.650000,1981.00000
25%,48.390000,2000.00000
50%,50.935000,2010.50000
75%,53.247500,2015.75000
max,54.230000,2020.00000


In [ ]:
reloaded_df.shape

(6, 5)

In [ ]:
reloaded_df.isna().sum()

,0
title,0
price_gbp,0
rating_word,0
author,0
first_publish_year,0
